# Compare `plt` vs `plt_`

Side-by-side 1D line-cut animation for two numpy output directories (e.g. different PEC settings).

```bash
# example: run with adi.output_dir = plt_ then copy/rename a prior run to plt/
make -j && ./main3d.gnu.ex inputs
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

from IPython.display import Video, display

DIR_A = Path("plt")
DIR_B = Path("plt_")
LABEL_A = "plt"
LABEL_B = "plt_"
COMPONENTS = ["Ex", "Ey", "Ez", "Hx", "Hy", "Hz"]
AXIS_NAMES = ("x", "y", "z")
INPUTS_PATH = Path("inputs")


def parse_inputs(path=INPUTS_PATH):
    groups = {}
    for line in Path(path).read_text().splitlines():
        line = line.split("#", 1)[0].strip()
        if not line or "=" not in line:
            continue
        key, val = line.split("=", 1)
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if "." not in key:
            continue
        prefix, name = key.split(".", 1)
        groups.setdefault(prefix, {})[name] = val

    prefix = "adi" if "adi" in groups else "fdtd" if "fdtd" in groups else None
    if prefix is None:
        raise ValueError(f"No adi.* or fdtd.* keys found in {path}")

    def _num(s, default=None):
        if s is None:
            return default
        return float(s) if any(ch in s for ch in ".eE") else int(s)

    raw = groups[prefix]
    return {
        "prefix": prefix,
        "ic_dir": int(_num(raw.get("ic_dir"), 2)),
        "ic_pol": int(_num(raw.get("ic_pol"), 0)),
        "pec_normal": int(_num(raw.get("pec_normal"), -1)),
        "pec_location": int(_num(raw.get("pec_location", 0), 0)),
    }


def list_steps(output_dir):
    steps = []
    for meta_path in sorted(output_dir.glob("step_*_meta.json")):
        meta = json.loads(meta_path.read_text())
        steps.append((meta["step"], meta_path))
    return steps


def load_step(meta_path):
    meta = json.loads(Path(meta_path).read_text())
    bin_path = Path(meta_path).with_name(Path(meta_path).name.replace("_meta.json", "_fields.bin"))
    fields = np.fromfile(bin_path, dtype=np.float64).reshape(meta["shape"])
    return fields, meta


def load_series(output_dir):
    series = []
    for step, meta_path in list_steps(output_dir):
        fields, meta = load_step(meta_path)
        series.append((meta["step"], meta["time"], fields, meta))
    return series


def coord_axes(meta):
    nx, ny, nz, _ = meta["shape"]
    prob_lo = np.array(meta["prob_lo"])
    prob_hi = np.array(meta["prob_hi"])
    x = np.linspace(prob_lo[0], prob_hi[0], nx, endpoint=False) + 0.5 * (prob_hi[0] - prob_lo[0]) / nx
    y = np.linspace(prob_lo[1], prob_hi[1], ny, endpoint=False) + 0.5 * (prob_hi[1] - prob_lo[1]) / ny
    z = np.linspace(prob_lo[2], prob_hi[2], nz, endpoint=False) + 0.5 * (prob_hi[2] - prob_lo[2]) / nz
    return x, y, z


def line_cut(fields, meta, line_axis, comp_idx):
    x, y, z = coord_axes(meta)
    ix, iy, iz = len(x) // 2, len(y) // 2, len(z) // 2
    if line_axis == "z":
        return z, fields[ix, iy, :, comp_idx]
    if line_axis == "y":
        return y, fields[ix, :, iz, comp_idx]
    return x, fields[:, iy, iz, comp_idx]


def active_ic_components(cfg):
    """Return (E_component_name, H_component_name) for the seeded IC."""
    pol = cfg["ic_pol"]
    bdir = 3 - cfg["ic_dir"] - pol
    return COMPONENTS[pol], COMPONENTS[3 + bdir]


IC_CFG = parse_inputs()
PLOT_H = True  # True: plot h_ic; False: plot e_ic
e_ic, h_ic = active_ic_components(IC_CFG)
LINE_AXIS = AXIS_NAMES[IC_CFG["ic_dir"]]
COMP = h_ic if PLOT_H else e_ic
comp_idx = COMPONENTS.index(COMP)

series_a = load_series(DIR_A)
series_b = load_series(DIR_B)
steps_a = {s: i for i, (s, *_) in enumerate(series_a)}
steps_b = {s: i for i, (s, *_) in enumerate(series_b)}
common_steps = sorted(set(steps_a) & set(steps_b))
if not common_steps:
    raise FileNotFoundError(f"No common steps between {DIR_A} and {DIR_B}")

print(f"{LABEL_A}: {len(series_a)} dumps in {DIR_A.resolve()}")
print(f"{LABEL_B}: {len(series_b)} dumps in {DIR_B}")
print(f"common steps: {len(common_steps)} ({common_steps[0]} .. {common_steps[-1]})")
print(
    f"plotting {COMP} along {LINE_AXIS}; E={e_ic}, H={h_ic}, "
    f"ic_dir={IC_CFG['ic_dir']}, pec_normal={IC_CFG['pec_normal']}, pec_location={IC_CFG['pec_location']}"
)

In [ ]:
OFFSET = 64  # shift plt_ by this many line-axis cell indices (periodic wrap)

def periodic_offset_line(coord, line, offset):
    """Shift line by index offset with periodic wrap; return sorted (coord, line) for plotting."""
    n = len(coord)
    dcoord = coord[1] - coord[0] if n > 1 else 1.0
    period = n * dcoord
    c0 = coord[0]
    coord_off = c0 + (coord - c0 + offset * dcoord) % period
    order = np.argsort(coord_off)
    return coord_off[order], line[order]


def periodic_align_on_grid(coord, line, offset):
    """Align plt_ onto coord grid: value at coord[i] is line[(i - offset) % n]."""
    return coord, np.roll(line, offset)


meta0 = series_a[0][3]
coord_a, _ = line_cut(series_a[0][2], meta0, LINE_AXIS, comp_idx)
dcoord = coord_a[1] - coord_a[0] if len(coord_a) > 1 else 1.0

lines_a = []
lines_b = []
lines_b_plot = []
lines_b_on_a = []
times = []
for step in common_steps:
    _, time_a, fields_a, meta_a = series_a[steps_a[step]]
    _, time_b, fields_b, meta_b = series_b[steps_b[step]]
    _, la = line_cut(fields_a, meta_a, LINE_AXIS, comp_idx)
    _, lb = line_cut(fields_b, meta_b, LINE_AXIS, comp_idx)
    coord_b, lb_plot = periodic_offset_line(coord_a, lb, OFFSET)
    _, lb_on_a = periodic_align_on_grid(coord_a, lb, OFFSET)
    lines_a.append(la)
    lines_b.append(lb)
    lines_b_plot.append(lb_plot)
    lines_b_on_a.append(lb_on_a)
    times.append(time_a)

diff_rms = [
    float(np.sqrt(np.mean((a - b) ** 2)))
    for a, b in zip(lines_a, lines_b_on_a)
]
print(f"RMS({LABEL_A} - {LABEL_B}): min={min(diff_rms):.2e}, max={max(diff_rms):.2e}")
if OFFSET != 0:
    print(f"plt_ shifted by {OFFSET} cells with periodic wrap ({OFFSET * dcoord:.3e} {LINE_AXIS})")

ymin = min(min(l.min() for l in lines_a), min(l.min() for l in lines_b_plot))
ymax = max(max(l.max() for l in lines_a), max(l.max() for l in lines_b_plot))
pad = 0.05 * max(abs(ymin), abs(ymax), 1e-12)
ylim = (ymin - pad, ymax + pad)

OUTPUT_WEBM = Path("plt") / f"{COMP.lower()}_line_{LINE_AXIS}_compare.webm"
FPS = 5

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, constrained_layout=True)
ax_top, ax_bot = axes
(line_a,) = ax_top.plot(coord_a, lines_a[0], "-", lw=1.5, label=LABEL_A)
(line_b,) = ax_top.plot(coord_b, lines_b_plot[0], "--", lw=1.5, label=LABEL_B)
(line_diff,) = ax_bot.plot(
    coord_a, lines_a[0] - lines_b_on_a[0], "-", lw=1.2, color="tab:red", label=f"{LABEL_A} - {LABEL_B}"
)
ax_top.legend(loc="upper right")
ax_bot.legend(loc="upper right")
ax_top.set_ylabel(COMP)
ax_bot.set_ylabel("difference")
xlabel = LINE_AXIS if OFFSET == 0 else f"{LINE_AXIS} ({LABEL_A}; {LABEL_B} + {OFFSET} cells, periodic)"
ax_bot.set_xlabel(xlabel)
ax_top.set_xlim(coord_a[0], coord_a[-1])
ax_bot.set_xlim(coord_a[0], coord_a[-1])
ax_top.set_ylim(ylim)
title_top = ax_top.set_title("")
title_bot = ax_bot.set_title("")
ax_top.grid(True, alpha=0.3)
ax_bot.grid(True, alpha=0.3)


def update(i):
    coord_b, lb_plot = periodic_offset_line(coord_a, lines_b[i], OFFSET)
    line_a.set_data(coord_a, lines_a[i])
    line_b.set_data(coord_b, lb_plot)
    line_diff.set_data(coord_a, lines_a[i] - lines_b_on_a[i])
    step = common_steps[i]
    title_top.set_text(f"{COMP} along {LINE_AXIS} at center  step {step}  t = {times[i]:.3e} s")
    title_bot.set_text(f"RMS diff = {diff_rms[i]:.2e}")
    return line_a, line_b, line_diff, title_top, title_bot


anim = animation.FuncAnimation(fig, update, frames=len(common_steps), interval=1000 // FPS, blit=False)
writer = animation.FFMpegWriter(fps=FPS, codec="libvpx-vp9", metadata={"title": f"{COMP} compare"})
anim.save(str(OUTPUT_WEBM), writer=writer)
plt.close(fig)

display(Video(str(OUTPUT_WEBM.resolve())))
print(f"Saved {OUTPUT_WEBM.resolve()}")